In [8]:
from pathlib import Path
from urllib.parse import quote
import requests
from datetime import datetime
import calendar

BASE = "https://coastwatch.noaa.gov/erddap/griddap"

DATASETS = {
    "sst": {
        "dataset_id": "noaacrwsstDaily",
        "variable": "analysed_sst",
    },
    "sst_anomaly": {
        "dataset_id": "noaacrwsstanomalyDaily",
        "variable": "sea_surface_temperature_anomaly",
    },
    "hotspot": {
        "dataset_id": "noaacrwhotspotDaily",
        "variable": "hotspot",
    },
    "dhw": {
        "dataset_id": "noaacrwdhwDaily",
        "variable": "degree_heating_week",
    },
    # "alert_level": {
    #     "dataset_id": "noaacrwbaa7dDaily",
    #     "variable": "bleaching_alert_area",
    # },
}

# Great Barrier Reef bounding box
LAT_MIN = -24.5
LAT_MAX = -10.0
LON_MIN = 142.0
LON_MAX = 154.0

START_YEAR = 2025
END_YEAR = 2025

OUTDIR = Path("crw_gbr_nc_yearly")
OUTDIR.mkdir(parents=True, exist_ok=True)


def build_griddap_nc_url(base, dataset_id, variable, start_dt, end_dt,
                         lat_min, lat_max, lon_min, lon_max):
    query = (
        f"{variable}"
        f"[({start_dt}):1:({end_dt})]"
        f"[({lat_min}):1:({lat_max})]"
        f"[({lon_min}):1:({lon_max})]"
    )
    return f"{base}/{dataset_id}.nc?{quote(query, safe='[]():,')}"


def download_file(url, output_path, chunk_size=1024 * 1024):
    print(f"Downloading -> {output_path.name}")
    with requests.get(url, stream=True, timeout=(30, 1800)) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    print(f"Saved: {output_path}")


def yearly_ranges(start_year, end_year):
    for year in range(start_year, end_year + 1):
        start_dt = f"{year}-01-01T12:00:00Z"
        end_dt = f"{year}-12-31T12:00:00Z"
        yield year, start_dt, end_dt

def monthly_ranges(start_year, end_year):
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            last_day = calendar.monthrange(year, month)[1]
            start_dt = f"{year}-{month:02d}-01T12:00:00Z"
            end_dt = f"{year}-{month:02d}-{last_day:02d}T12:00:00Z"
            yield year, month, start_dt, end_dt

def custom_ranges():
    yield 2026, 4, "2026-04-01T12:00:00Z", "2026-04-12T12:00:00Z"

def main():
    for product_name, meta in DATASETS.items():
        dataset_id = meta["dataset_id"]
        variable = meta["variable"]

        product_dir = OUTDIR / product_name
        product_dir.mkdir(exist_ok=True)

        # for year, start_dt, end_dt in yearly_ranges(START_YEAR, END_YEAR):
        #     outfile = product_dir / f"{product_name}_gbr_{year}.nc"
        # for year, month, start_dt, end_dt in monthly_ranges(START_YEAR, END_YEAR):
        for year, month, start_dt, end_dt in custom_ranges():
            outfile = product_dir / f"{product_name}_gbr_{year}_{month:02d}.nc"
            if outfile.exists():
                print(f"Skipping existing file: {outfile}")
                continue

            url = build_griddap_nc_url(
                base=BASE,
                dataset_id=dataset_id,
                variable=variable,
                start_dt=start_dt,
                end_dt=end_dt,
                lat_min=LAT_MIN,
                lat_max=LAT_MAX,
                lon_min=LON_MIN,
                lon_max=LON_MAX,
            )

            try:
                download_file(url, outfile)
            except requests.HTTPError as e:
                print(f"Failed for {product_name} {year}: {e}")
            except Exception as e:
                print(f"Unexpected error for {product_name} {year}: {e}")


if __name__ == "__main__":
    main()

Saved: crw_gbr_nc_yearly/sst/sst_gbr_2026_04.nc
Saved: crw_gbr_nc_yearly/sst_anomaly/sst_anomaly_gbr_2026_04.nc
Skipping existing file: crw_gbr_nc_yearly/hotspot/hotspot_gbr_2026_04.nc
Saved: crw_gbr_nc_yearly/dhw/dhw_gbr_2026_04.nc


In [20]:
"""
Download NOAA Coral Reef Watch daily source NetCDF files from ERDDAP /files/,
subset each file locally to a target bounding box, and save the subset as .nc.

Tested design assumptions:
- NOAA CoastWatch ERDDAP exposes CRW source files under /erddap/files/{dataset_id}/{year}/
- Source file variable/coordinate names may differ from griddap names, so we infer them
- We process one file at a time to be gentle on the server
"""

from __future__ import annotations

import os
import re
import time
import shutil
import logging
from pathlib import Path
from datetime import date, datetime, timedelta
from typing import Iterable, Optional, Tuple, Dict, List
from urllib.parse import urljoin

import requests
import xarray as xr
from bs4 import BeautifulSoup


# ----------------------------
# User configuration
# ----------------------------

BASE_FILES_URL = "https://coastwatch.noaa.gov/erddap/files"

PRODUCTS: Dict[str, str] = {
    "sst": "noaacrwsstDaily",
    "sst_anomaly": "noaacrwsstanomalyDaily",
    "hotspot": "noaacrwhotspotDaily",
    "dhw": "noaacrwdhwDaily",
    "alert_level": "noaacrwbaa7dDaily",
}

# Great Barrier Reef bounding box
LAT_MIN = -24.5
LAT_MAX = -10.0
LON_MIN = 142.0
LON_MAX = 154.0

START_DATE = date(2015, 1, 1)
END_DATE = date(2025, 12, 31)

# Where to store outputs
OUT_ROOT = Path("crw_gbr_subsets")

# Optional temp directory for full daily files before subsetting
TMP_ROOT = Path("tmp_crw_downloads")

# Be polite to the public server
REQUEST_DELAY_SECONDS = 2.0
HTTP_TIMEOUT = 120
MAX_RETRIES = 4
BACKOFF_SECONDS = 5

# Save compression
NETCDF_ENGINE = "netcdf4"   # requires netCDF4 installed
COMPRESS = True

# If True, keep the original full daily file after subsetting
KEEP_FULL_SOURCE_FILE = False

# If True, skip files whose subset already exists
SKIP_EXISTING = True


# ----------------------------
# Logging
# ----------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("crw_downloader")


# ----------------------------
# Helpers
# ----------------------------

DATE_RE = re.compile(r"(\d{8})\.nc$", re.IGNORECASE)


def daterange(start: date, end: date) -> Iterable[date]:
    current = start
    while current <= end:
        yield current
        current += timedelta(days=1)


def year_span(start: date, end: date) -> Iterable[int]:
    for y in range(start.year, end.year + 1):
        yield y


def extract_date_from_filename(filename: str) -> Optional[date]:
    m = DATE_RE.search(filename)
    if not m:
        return None
    return datetime.strptime(m.group(1), "%Y%m%d").date()


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def request_with_retries(
    session: requests.Session,
    url: str,
    stream: bool = False,
    timeout: int = HTTP_TIMEOUT,
) -> requests.Response:
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, timeout=timeout, stream=stream)
            resp.raise_for_status()
            return resp
        except Exception as exc:
            last_exc = exc
            wait = BACKOFF_SECONDS * attempt
            logger.warning("Request failed (%s/%s): %s", attempt, MAX_RETRIES, url)
            logger.warning("Error: %s", exc)
            if attempt < MAX_RETRIES:
                logger.info("Sleeping %.1f seconds before retry", wait)
                time.sleep(wait)
    raise last_exc  # type: ignore[misc]


def list_year_files(session: requests.Session, dataset_id: str, year: int) -> List[Tuple[date, str, str]]:
    """
    Parse the ERDDAP /files/{dataset_id}/{year}/ directory listing and return:
    [(file_date, filename, absolute_url), ...]
    """
    year_url = f"{BASE_FILES_URL}/{dataset_id}/{year}/"
    resp = request_with_retries(session, year_url, stream=False)
    soup = BeautifulSoup(resp.text, "html.parser")

    found: List[Tuple[date, str, str]] = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(strip=True)

        # Prefer the visible link text if it ends with .nc, otherwise href
        candidate = text if text.lower().endswith(".nc") else href
        if not candidate.lower().endswith(".nc"):
            continue

        file_date = extract_date_from_filename(candidate)
        if file_date is None:
            continue

        abs_url = urljoin(year_url, href)
        found.append((file_date, candidate, abs_url))

    found.sort(key=lambda x: x[0])
    return found


def infer_coord_name(ds: xr.Dataset, candidates: List[str], role: str) -> str:
    """
    Find a coordinate/dimension name robustly across NOAA source files.
    """
    lower_map = {name.lower(): name for name in ds.coords}
    lower_dims = {name.lower(): name for name in ds.dims}
    lower_vars = {name.lower(): name for name in ds.variables}

    for c in candidates:
        if c in lower_map:
            return lower_map[c]
        if c in lower_dims:
            return lower_dims[c]
        if c in lower_vars:
            return lower_vars[c]

    raise KeyError(
        f"Could not infer {role} coordinate. "
        f"Available coords={list(ds.coords)}, dims={list(ds.dims)}, vars={list(ds.variables)}"
    )


def subset_spatial(
    ds: xr.Dataset,
    lat_min: float,
    lat_max: float,
    lon_min: float,
    lon_max: float,
) -> xr.Dataset:
    lat_name = infer_coord_name(ds, ["lat", "latitude", "y"], "latitude")
    lon_name = infer_coord_name(ds, ["lon", "longitude", "x"], "longitude")

    lat_vals = ds[lat_name].values
    lon_vals = ds[lon_name].values

    if lat_vals[0] <= lat_vals[-1]:
        ds = ds.sel({lat_name: slice(lat_min, lat_max)})
    else:
        ds = ds.sel({lat_name: slice(lat_max, lat_min)})

    if lon_vals[0] <= lon_vals[-1]:
        ds = ds.sel({lon_name: slice(lon_min, lon_max)})
    else:
        ds = ds.sel({lon_name: slice(lon_max, lon_min)})

    return ds


def build_encoding(ds: xr.Dataset) -> Dict[str, dict]:
    """
    Compression settings for data variables only.
    """
    encoding = {}
    for var in ds.data_vars:
        if COMPRESS:
            encoding[var] = {"zlib": True, "complevel": 4}
        else:
            encoding[var] = {}
    return encoding


def safe_open_dataset(path: Path) -> xr.Dataset:
    """
    Open lazily; close explicitly with context manager semantics.
    """
    return xr.open_dataset(path)


def download_file(session: requests.Session, url: str, dest: Path) -> None:
    ensure_dir(dest.parent)
    with request_with_retries(session, url, stream=True) as resp:
        with open(dest, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


def process_one_file(
    session: requests.Session,
    product_name: str,
    dataset_id: str,
    file_date: date,
    filename: str,
    source_url: str,
    out_root: Path,
    tmp_root: Path,
    lat_min: float,
    lat_max: float,
    lon_min: float,
    lon_max: float,
) -> None:
    year_dir = out_root / product_name / str(file_date.year)
    ensure_dir(year_dir)

    subset_out = year_dir / filename
    if SKIP_EXISTING and subset_out.exists():
        logger.info("Skipping existing subset: %s", subset_out)
        return

    tmp_dir = tmp_root / product_name / str(file_date.year)
    ensure_dir(tmp_dir)
    full_tmp = tmp_dir / filename

    logger.info("Downloading %s | %s", product_name, filename)
    download_file(session, source_url, full_tmp)

    try:
        with safe_open_dataset(full_tmp) as ds:
            ds_subset = subset_spatial(ds, lat_min, lat_max, lon_min, lon_max)

            # Preserve useful provenance
            ds_subset.attrs["subset_bbox"] = (
                f"lat=[{lat_min},{lat_max}], lon=[{lon_min},{lon_max}]"
            )
            ds_subset.attrs["source_file_url"] = source_url
            ds_subset.attrs["source_dataset_id"] = dataset_id
            ds_subset.attrs["subset_created_utc"] = datetime.utcnow().isoformat() + "Z"

            encoding = build_encoding(ds_subset)
            ensure_dir(subset_out.parent)
            ds_subset.to_netcdf(subset_out, engine=NETCDF_ENGINE, encoding=encoding)

        logger.info("Saved subset: %s", subset_out)

    finally:
        if not KEEP_FULL_SOURCE_FILE and full_tmp.exists():
            try:
                full_tmp.unlink()
            except Exception:
                logger.warning("Could not delete temp file: %s", full_tmp)


def combine_year_to_single_file(product_dir: Path, year: int, out_file: Path) -> None:
    """
    Optional helper: merge all daily subset files in a year into one NetCDF.
    This is local-only and does not hit NOAA.
    """
    files = sorted((product_dir / str(year)).glob("*.nc"))
    if not files:
        raise FileNotFoundError(f"No files found under {product_dir / str(year)}")

    ds = xr.open_mfdataset(
        [str(f) for f in files],
        combine="by_coords",
        parallel=False,
    )
    ds.to_netcdf(out_file, engine=NETCDF_ENGINE, encoding=build_encoding(ds))
    ds.close()


# ----------------------------
# Main workflow
# ----------------------------

def run() -> None:
    ensure_dir(OUT_ROOT)
    ensure_dir(TMP_ROOT)

    with requests.Session() as session:
        session.headers.update(
            {
                "User-Agent": "crw-gbr-downloader/1.0 (sequential polite downloader)",
                "Accept": "*/*",
            }
        )

        for product_name, dataset_id in PRODUCTS.items():
            logger.info("=== Product: %s (%s) ===", product_name, dataset_id)

            for year in year_span(START_DATE, END_DATE):
                logger.info("Listing year directory: %s / %s", dataset_id, year)
                try:
                    year_files = list_year_files(session, dataset_id, year)
                except Exception as exc:
                    logger.error("Failed to list %s/%s: %s", dataset_id, year, exc)
                    continue

                # Restrict to target date window
                wanted = [
                    (file_date, filename, url)
                    for file_date, filename, url in year_files
                    if START_DATE <= file_date <= END_DATE
                ]

                logger.info(
                    "Found %d files in requested range for %s %s",
                    len(wanted), product_name, year
                )

                for file_date, filename, url in wanted:
                    try:
                        process_one_file(
                            session=session,
                            product_name=product_name,
                            dataset_id=dataset_id,
                            file_date=file_date,
                            filename=filename,
                            source_url=url,
                            out_root=OUT_ROOT,
                            tmp_root=TMP_ROOT,
                            lat_min=LAT_MIN,
                            lat_max=LAT_MAX,
                            lon_min=LON_MIN,
                            lon_max=LON_MAX,
                        )
                    except Exception as exc:
                        logger.exception(
                            "Failed processing %s %s (%s): %s",
                            product_name,
                            file_date.isoformat(),
                            filename,
                            exc,
                        )

                    # polite pause between source-file downloads
                    # time.sleep(REQUEST_DELAY_SECONDS)


if __name__ == "__main__":
    run()

2026-03-30 18:08:22,292 | INFO | === Product: sst (noaacrwsstDaily) ===
2026-03-30 18:08:22,293 | INFO | Listing year directory: noaacrwsstDaily / 2015
2026-03-30 18:08:23,540 | INFO | Found 365 files in requested range for sst 2015
2026-03-30 18:08:23,540 | INFO | Skipping existing subset: crw_gbr_subsets/sst/2015/coraltemp_v3.1_20150101.nc
2026-03-30 18:08:23,541 | INFO | Skipping existing subset: crw_gbr_subsets/sst/2015/coraltemp_v3.1_20150102.nc
2026-03-30 18:08:23,541 | INFO | Skipping existing subset: crw_gbr_subsets/sst/2015/coraltemp_v3.1_20150103.nc
2026-03-30 18:08:23,541 | INFO | Skipping existing subset: crw_gbr_subsets/sst/2015/coraltemp_v3.1_20150104.nc
2026-03-30 18:08:23,542 | INFO | Skipping existing subset: crw_gbr_subsets/sst/2015/coraltemp_v3.1_20150105.nc
2026-03-30 18:08:23,542 | INFO | Skipping existing subset: crw_gbr_subsets/sst/2015/coraltemp_v3.1_20150106.nc
2026-03-30 18:08:23,542 | INFO | Skipping existing subset: crw_gbr_subsets/sst/2015/coraltemp_v3.1_20

KeyboardInterrupt: 